<a href="https://colab.research.google.com/github/5atish/TrainingCourse/blob/main/Hugging%20Face%20Agents%20Course%20/Unit%203.%20Use%20Case%20for%20Agentic%20RAG/Creating_a_RAG_Tool_for_Guest_Stories.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import datasets
from langchain_core.documents import Document

# Load the dataset
guest_dataset = datasets.load_dataset("agents-course/unit3-invitees", split="train")

# Convert dataset entries into Document objects
docs = [
    Document(
        page_content="\n".join([
            f"Name: {guest['name']}",
            f"Relation: {guest['relation']}",
            f"Description: {guest['description']}",
            f"Email: {guest['email']}"
        ]),
        metadata={"name": guest["name"]}
    )
    for guest in guest_dataset
]


In [2]:
# !pip install llama_index

In [3]:
import datasets
from llama_index.core.schema import Document

# Load the dataset
guest_dataset = datasets.load_dataset("agents-course/unit3-invitees", split="train")

# Convert dataset entries into Document objects
docs = [
    Document(
        text="\n".join([
            f"Name: {guest_dataset['name'][i]}",
            f"Relation: {guest_dataset['relation'][i]}",
            f"Description: {guest_dataset['description'][i]}",
            f"Email: {guest_dataset['email'][i]}"
        ]),
        metadata={"name": guest_dataset['name'][i]}
    )
    for i in range(len(guest_dataset))
]

In [4]:
import datasets
from langchain_core.documents import Document

# Load the dataset
guest_dataset = datasets.load_dataset("agents-course/unit3-invitees", split="train")

# Convert dataset entries into Document objects
docs = [
    Document(
        page_content="\n".join([
            f"Name: {guest['name']}",
            f"Relation: {guest['relation']}",
            f"Description: {guest['description']}",
            f"Email: {guest['email']}"
        ]),
        metadata={"name": guest["name"]}
    )
    for guest in guest_dataset
]

In [5]:
  # !pip install llama-index-retrievers-bm25

In [6]:
from smolagents import Tool
from langchain_community.retrievers import BM25Retriever

class GuestInfoRetrieverTool(Tool):
    name = "guest_info_retriever"
    description = "Retrieves detailed information about gala guests based on their name or relation."
    inputs = {
        "query": {
            "type": "string",
            "description": "The name or relation of the guest you want information about."
        }
    }
    output_type = "string"

    def __init__(self, docs):
        self.is_initialized = False
        self.retriever = BM25Retriever.from_documents(docs)

    def forward(self, query: str):
        results = self.retriever.invoke(query)
        if results:
            return "\n\n".join([doc.page_content for doc in results[:3]])
        else:
            return "No matching guest information found."

# Initialize the tool
guest_info_tool = GuestInfoRetrieverTool(docs)

/tmp/ipykernel_9941/3722892848.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [7]:
from llama_index.core.tools import FunctionTool
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.schema import Document as LlamaIndexDocument # Alias for clarity

# Assuming 'docs' currently holds langchain_core.documents.Document objects from previous cells.
# Convert them to llama_index.core.schema.Document objects.
llama_docs = [
    LlamaIndexDocument(text=doc.page_content, metadata=doc.metadata)
    for doc in docs
]

bm25_retriever = BM25Retriever.from_defaults(nodes=llama_docs)

def get_guest_info_retriever(query: str) -> str:
    """Retrieves detailed information about gala guests based on their name or relation."""
    results = bm25_retriever.retrieve(query)
    if results:
        # Retrieved documents will be LlamaIndexDocument objects which have a 'text' attribute
        return "\n\n".join([doc.text for doc in results[:3]])
    else:
        return "No matching guest information found."

# Initialize the tool
guest_info_tool = FunctionTool.from_defaults(get_guest_info_retriever)

DEBUG:bm25s:Building index from IDs objects


In [13]:
class GuestInfoRetrieverTool(Tool):
    name = "guest_info_retriever"
    description = "Retrieves detailed information about gala guests based on their name or relation."
    inputs = {
        "query": {
            "type": "string",
            "description": "The name or relation of the guest you want information about."
        }
    }
    output_type = "string"

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)
        self.retriever = BM25Retriever.from_documents(docs)

    def forward(self, query: str):
        results = self.retriever.invoke(query)
        if results:
            return "\n\n".join([doc.page_content for doc in results[:3]])
        else:
            return "No matching guest information found."

In [14]:
alfred_guest_info_tool = GuestInfoRetrieverTool(docs)

alfred = CodeAgent(tools=[alfred_guest_info_tool], model=model)

response = alfred.run("Tell me about our guest named 'Lady Ada Lovelace'.")
print("🎩 Alfred's Response:")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Tell me about our guest named 'Lady Ada Lovelace'.                                                              │
│                                                                                                                 │
╰─ OpenAIModel - openai/gpt-oss-120b ─────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  info = guest_info_retriever(query="Lady Ada Lovelace")                                                           
  final_answer(info)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for 
her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work
on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com

Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless 
energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about 
pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com

[Step 1: Duration 0.85 seconds| Input tokens: 2,094 | Output tokens: 231]

🎩 Alfred's Response:
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com

Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com


In [15]:
from smolagents import CodeAgent, OpenAIServerModel, Tool
from langchain_community.retrievers import BM25Retriever
from google.colab import userdata
from smolagents import OpenAIServerModel
import os


os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
# Redefine the GuestInfoRetrieverTool class from cell 3JXT_R5vyO7G
# to ensure it's available and correctly typed for smolagents.CodeAgent
class GuestInfoRetrieverTool(Tool):
    name = "guest_info_retriever"
    description = "Retrieves detailed information about gala guests based on their name or relation."
    inputs = {
        "query": {
            "type": "string",
            "description": "The name or relation of the guest you want information about."
        }
    }
    output_type = "string"

    def __init__(self, docs):
        self.retriever = BM25Retriever.from_documents(docs)

    def forward(self, query: str):
        results = self.retriever.invoke(query)
        if results:
            return "\n\n".join([doc.page_content for doc in results[:3]])
        else:
            return "No matching guest information found."

# Initialize the correct tool instance specifically for Alfred
alfred_guest_info_tool = GuestInfoRetrieverTool(docs)

# Initialize the Hugging Face model with the correct parameter: model_id
# Using 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' as a small, explicitly chat-tuned model.
model = OpenAIServerModel(model_id="openai/gpt-oss-120b",
                             api_base="https://api.groq.com/openai/v1",
                             api_key=os.environ["GROQ_API_KEY"])

# Create Alfred, our gala agent, with the correctly typed guest info tool
alfred = CodeAgent(tools=[alfred_guest_info_tool], model=model)

# Example query Alfred might receive during the gala
response = alfred.run("Tell me about our guest named 'Lady Ada Lovelace'.")

print("🎩 Alfred's Response:")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Tell me about our guest named 'Lady Ada Lovelace'.                                                              │
│                                                                                                                 │
╰─ OpenAIModel - openai/gpt-oss-120b ─────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
                                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: None

[Step 1: Duration 0.66 seconds| Input tokens: 2,094 | Output tokens: 171]

...

[Step 3: Duration 18.61 seconds| Input tokens: 6,456 | Output tokens: 567]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 
'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "python", "arguments": info = 
guest_info_retriever(query="Lady Ada Lovelace")\nprint(info)}'}}

[Step 4: Duration 18.83 seconds]

AgentGenerationError: Error in generating model output:
Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "python", "arguments": info = guest_info_retriever(query="Lady Ada Lovelace")\nprint(info)}'}}

In [16]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.0 MB/s eta 0:00:00


In [18]:
from langchain_core.tools import tool
from langchain_community.retrievers import BM25Retriever

retriever = BM25Retriever.from_documents(docs)

@tool
def guest_info_tool(query: str) -> str:
    """Retrieves detailed information about gala guests based on their name or relation."""
    results = retriever.invoke(query)
    if results:
        return "\n\n".join([doc.page_content for doc in results[:3]])
    else:
        return "No matching guest information found."

In [19]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage, AnyMessage
from typing import TypedDict, Annotated
import os

# Generate the chat interface, including the tools
llm = ChatGroq(
    model="openai/gpt-oss-120b",  # using the model that's actually live on Groq, per your last fix
    groq_api_key=os.environ["GROQ_API_KEY"],
)

tools = [guest_info_tool]
chat_with_tools = llm.bind_tools(tools)

# Generate the AgentState and Agent graph
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def assistant(state: AgentState):
    return {
        "messages": [chat_with_tools.invoke(state["messages"])],
    }

## The graph
builder = StateGraph(AgentState)

builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    tools_condition,
)
builder.add_edge("tools", "assistant")

alfred = builder.compile()

messages = [HumanMessage(content="Tell me about our guest named 'Lady Ada Lovelace'.")]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
**Guest Profile: Lady Ada Lovelace**

- **Relation:** Best friend  
- **Description:** Lady Ada Lovelace is an esteemed mathematician and a true pioneer in the world of computing. She is celebrated as the first computer programmer for her visionary notes on Charles Babbame’s Analytical Engine, where she described how the machine could be used to manipulate symbols and not just numbers. Her insight into the potential of machines to process information laid the groundwork for modern computer science.  
- **Email:** ada.lovelace@example.com  

If you’d like to arrange a meeting, send her a note, or need any additional details (e.g., preferred topics of conversation, dietary preferences, etc.), just let me know!
